# Makemore Part 4: 역전파 닌자 되기

> 📺 Andrej Karpathy - "Becoming a Backprop Ninja"
> 🔗 https://www.youtube.com/watch?v=q8SA3rM6ckI

**목표**: `loss.backward()` 없이 수동으로 모든 기울기를 계산

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

---
## 데이터 & 모델 준비

In [2]:
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
block_size = 3
n_embd = 10
n_hidden = 64  # 작게 설정 (디버깅 용이)

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
Xtr, Ytr = build_dataset(words[:n1])

In [3]:
# 파라미터 초기화
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / (block_size * n_embd)**0.5
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0
# BatchNorm
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

print(f'총 파라미터 수: {sum(p.nelement() for p in parameters)}')

총 파라미터 수: 4137


---
## 📺 10:00 - 30:00 | 순전파 (각 단계 분리)

In [4]:
batch_size = 32
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

# 순전파 (각 단계를 변수로 저장)
emb = C[Xb]                                       # 임베딩 lookup
embcat = emb.view(emb.shape[0], -1)               # 연결
hprebn = embcat @ W1 + b1                          # 선형 변환
# BatchNorm
bnmeani = hprebn.mean(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1/(batch_size-1) * bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias                  # scale & shift
# 활성화
h = torch.tanh(hpreact)
logits = h @ W2 + b2                               # 출력층
# loss
loss = F.cross_entropy(logits, Yb)

print(f'loss = {loss.item():.4f}')

loss = 3.3105


---
## 📺 30:00 - 01:30:00 | 수동 역전파

PyTorch backward와 비교하며 검증

In [5]:
# 먼저 PyTorch의 자동 역전파로 정답 구하기
for p in parameters:
    p.grad = None
for t in [logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, hprebn, embcat, emb]:
    t.retain_grad()
loss.backward()
print('PyTorch backward 완료')

PyTorch backward 완료


In [6]:
# 검증 함수
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff:.6f}')

In [7]:
# 수동 역전파: softmax + cross entropy
# dlogits = softmax 확률 - 정답 one-hot
dlogits = F.softmax(logits, dim=1)
dlogits[range(batch_size), Yb] -= 1
dlogits /= batch_size
cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 0.000000


In [8]:
# 행렬곱 역전파: logits = h @ W2 + b2
dh = dlogits @ W2.T         # dh = dlogits @ W2.T
dW2 = h.T @ dlogits         # dW2 = h.T @ dlogits
db2 = dlogits.sum(0)        # 브로드캐스팅의 역전파 = sum

cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

h               | exact: False | approximate: True  | maxdiff: 0.000000
W2              | exact: False | approximate: True  | maxdiff: 0.000000
b2              | exact: False | approximate: True  | maxdiff: 0.000000


In [9]:
# tanh 역전파: h = tanh(hpreact)
dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

hpreact         | exact: False | approximate: True  | maxdiff: 0.000000


In [10]:
# BatchNorm 역전파
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnbias = dhpreact.sum(0, keepdim=True)
dhprebn = bngain * bnvar_inv / batch_size * (
    batch_size * dhpreact 
    - dhpreact.sum(0) 
    - batch_size/(batch_size-1) * bnraw * (dhpreact * bnraw).sum(0)
)

cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('hprebn', dhprebn, hprebn)

bngain          | exact: False | approximate: True  | maxdiff: 0.000000
bnbias          | exact: False | approximate: True  | maxdiff: 0.000000
hprebn          | exact: False | approximate: True  | maxdiff: 0.000000


In [11]:
# 선형층 역전파: hprebn = embcat @ W1 + b1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

embcat          | exact: False | approximate: True  | maxdiff: 0.000000
W1              | exact: False | approximate: True  | maxdiff: 0.000000
b1              | exact: False | approximate: True  | maxdiff: 0.000000


In [12]:
# 임베딩 역전파
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

cmp('C', dC, C)

C               | exact: False | approximate: True  | maxdiff: 0.000000


---
## 결과

모든 기울기가 PyTorch의 자동 미분과 **일치**함을 확인!